In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

from mlforecast import MLForecast
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

try:
    from mlforecast.lag_transforms import RollingMean, RollingStd
    LAG_TRANSFORMS_API = "new"
except ModuleNotFoundError:
    from window_ops.rolling import rolling_mean, rolling_std
    LAG_TRANSFORMS_API = "old"

print(f"Libraries imported! (mlforecast lag_transforms API: {LAG_TRANSFORMS_API})")

# 1. Configuration
FREQUENCY = 'Daily'
SEASONALITY = 7
HORIZON = 14
RANDOM_STATE = 42
RF_N_SERIES_SAMPLE = 1200

data_path = Path('../data/M4')
train_file = data_path / f'{FREQUENCY}-train.csv'
test_file = data_path / f'{FREQUENCY}-test.csv'

# 2. Evaluation Functions
def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error (metrica ufficiale M4)"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return 100 * np.mean(diff)

def mase(y_true, y_pred, y_train, seasonality=1):
    """Mean Absolute Scaled Error"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_train = np.array(y_train)
    mae = np.mean(np.abs(y_true - y_pred))
    naive_mae = np.mean(np.abs(y_train[seasonality:] - y_train[:-seasonality]))
    if naive_mae == 0:
        return np.nan
    return mae / naive_mae

# 3. Data loading
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")
print(f"Series: {len(train_df)}")
print(f"Forecast horizon: {HORIZON} days")
print(f"Seasonality: {SEASONALITY} days")

id_to_idx = {sid: i for i, sid in enumerate(train_df.iloc[:, 0])}

# 4. Conversion to Long Format
def build_long_df(df, max_history=None):
    """Converte il dataframe wide M4 (una riga per serie) in formato long."""
    records = []
    for idx in tqdm(range(len(df)), desc="Building long format"):
        series_id = df.iloc[idx, 0]
        series = df.iloc[idx, 1:].dropna().values
        if max_history is not None and len(series) > max_history:
            series = series[-max_history:]
        n = len(series)
        if n == 0:
            continue
        records.append(pd.DataFrame({
            'unique_id': series_id,
            'ds': np.arange(1, n + 1),
            'y': series.astype(np.float32)
        }))
    return pd.concat(records, ignore_index=True)

train_long_full = build_long_df(train_df, max_history=None)
print(f"Righe totali: {len(train_long_full):,}")

# 5. Per-series normalization (z-score)
scale_stats = (
    train_long_full.groupby('unique_id')['y']
    .agg(y_mean='mean', y_std='std')
    .reset_index()
)

scale_stats['y_std'] = scale_stats['y_std'].replace(0, 1).fillna(1)

train_long_scaled = train_long_full.merge(scale_stats, on='unique_id', how='left')
train_long_scaled['y'] = (train_long_scaled['y'] - train_long_scaled['y_mean']) / train_long_scaled['y_std']
train_long_scaled = train_long_scaled[['unique_id', 'ds', 'y']]

print("Scaling applicato.")

# 6. Feature engineering
if LAG_TRANSFORMS_API == "new":
    lag_transforms = {
        1: [RollingMean(window_size=7), RollingMean(window_size=14),
            RollingMean(window_size=28), RollingStd(window_size=7)],
        7: [RollingMean(window_size=7)],
    }
else:
    lag_transforms = {
        1: [(rolling_mean, 7), (rolling_mean, 14), (rolling_mean, 28), (rolling_std, 7)],
        7: [(rolling_mean, 7)],
    }

LAGS = [1, 2, 7, 14, 21, 28]
print("Feature config pronta.")

# 7. Boosting Model Fitting 
mlf_boost = MLForecast(
    models={
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=300, num_leaves=31, learning_rate=0.05,
            min_child_samples=20, verbosity=-1, random_state=RANDOM_STATE,
        ),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, verbosity=0,
        ),
    },
    freq=1, lags=LAGS, lag_transforms=lag_transforms,
)

print("Fitting LightGBM + XGBoost...")
mlf_boost.fit(train_long_scaled, static_features=[])

print("Forecasting...")
boost_forecasts_scaled = mlf_boost.predict(HORIZON)

# 8. Random Forest Fitting
rng = np.random.RandomState(RANDOM_STATE)
rf_series_sample = rng.choice(train_df.iloc[:, 0].values, size=RF_N_SERIES_SAMPLE, replace=False)
train_long_rf = train_long_scaled[train_long_scaled['unique_id'].isin(rf_series_sample)]
print(f"Serie usate per il training RF: {train_long_rf['unique_id'].nunique()} ({len(train_long_rf):,} righe)")

mlf_rf = MLForecast(
    models={
        'RandomForest': RandomForestRegressor(
            n_estimators=100, max_depth=12, min_samples_leaf=5,
            n_jobs=-1, random_state=RANDOM_STATE,
        ),
    },
    freq=1, lags=LAGS, lag_transforms=lag_transforms,
)

print("Fitting Random Forest...")
mlf_rf.fit(train_long_rf, static_features=[])

print("Forecasting on all series (cross-learning)...")
rf_forecasts_scaled = mlf_rf.predict(h=HORIZON, new_df=train_long_scaled)

# 9. Inverse transform
def inverse_scale(forecasts_scaled, model_cols):
    merged = forecasts_scaled.merge(scale_stats, on='unique_id', how='left')
    for col in model_cols:
        merged[col] = merged[col] * merged['y_std'] + merged['y_mean']
    return merged

boost_model_cols = [c for c in boost_forecasts_scaled.columns if c not in ('unique_id', 'ds')]
rf_model_cols = [c for c in rf_forecasts_scaled.columns if c not in ('unique_id', 'ds')]

boost_forecasts = inverse_scale(boost_forecasts_scaled, boost_model_cols)
rf_forecasts = inverse_scale(rf_forecasts_scaled, rf_model_cols)

print("Inverse-transform completed.")

# 10. Evaluation
def evaluate_forecasts(forecasts_df, model_cols, label=""):
    scores = {m: {'smape': [], 'mase': []} for m in model_cols}
    for series_id, group in tqdm(forecasts_df.groupby('unique_id'), desc=f"Evaluating {label}"):
        idx = id_to_idx[series_id]
        train_series_full = train_df.iloc[idx, 1:].dropna().values
        test_series = test_df.iloc[idx, 1:].dropna().values

        group = group.sort_values('ds')
        h = min(len(test_series), len(group))
        y_true = test_series[:h]

        for model_col in model_cols:
            y_pred = group[model_col].values[:h]
            s = smape(y_true, y_pred)
            m = mase(y_true, y_pred, train_series_full, seasonality=1)
            if not np.isnan(s):
                scores[model_col]['smape'].append(s)
            if not np.isnan(m):
                scores[model_col]['mase'].append(m)
    return scores

boost_scores = evaluate_forecasts(boost_forecasts, boost_model_cols, label="LightGBM/XGBoost")
rf_scores = evaluate_forecasts(rf_forecasts, rf_model_cols, label="RandomForest")
all_scores = {**boost_scores, **rf_scores}

print("\n" + "="*80)
print(" ML MODELS RESULTS (scaled)")
print("="*80)
for model_col, s in all_scores.items():
    print(f"\n{model_col}:")
    print(f"   sMAPE: {np.mean(s['smape']):.4f}")
    print(f"   MASE:  {np.mean(s['mase']):.4f}")

In [ ]:
# --- canonical calib/test split
CONTEXT_LENGTH = 90  
MIN_SERIES_LENGTH = CONTEXT_LENGTH + HORIZON + 2  
N_SERIES_CALIB, N_SERIES_TEST = 500, 1000

lengths = train_df.iloc[:, 1:].notna().sum(axis=1)
filtered_ids = train_df.iloc[:, 0][lengths >= MIN_SERIES_LENGTH].tolist()
calib_ids = filtered_ids[:N_SERIES_CALIB]
test_ids = filtered_ids[N_SERIES_CALIB:N_SERIES_CALIB + N_SERIES_TEST]
exclude_ids = set(calib_ids) | set(test_ids)

# --- training pool
train_long_clean = train_long_scaled[~train_long_scaled['unique_id'].isin(exclude_ids)]

# --- refit Boosting 
mlf_boost_cal = MLForecast(
    models={
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=300, num_leaves=31, learning_rate=0.05,
            min_child_samples=20, verbosity=-1, random_state=RANDOM_STATE,
        ),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, verbosity=0,
        ),
    },
    freq=1, lags=LAGS, lag_transforms=lag_transforms,
)
print("Fitting LightGBM + XGBoost...")
mlf_boost_cal.fit(train_long_clean, static_features=[])

# --- refit Random Forest 
rng_cal = np.random.RandomState(RANDOM_STATE)
rf_pool_ids = np.array([sid for sid in train_df.iloc[:, 0].values if sid not in exclude_ids])
rf_series_sample_cal = rng_cal.choice(rf_pool_ids, size=min(RF_N_SERIES_SAMPLE, len(rf_pool_ids)), replace=False)
train_long_rf_cal = train_long_clean[train_long_clean['unique_id'].isin(rf_series_sample_cal)]

mlf_rf_cal = MLForecast(
    models={
        'RandomForest': RandomForestRegressor(
            n_estimators=100, max_depth=12, min_samples_leaf=5,
            n_jobs=-1, random_state=RANDOM_STATE,
        ),
    },
    freq=1, lags=LAGS, lag_transforms=lag_transforms,
)
print("Fitting Random...")
mlf_rf_cal.fit(train_long_rf_cal, static_features=[])

# --- calib: 
calib_long = train_long_scaled[train_long_scaled['unique_id'].isin(calib_ids)].sort_values(['unique_id', 'ds'])
rank_from_end = calib_long.groupby('unique_id').cumcount(ascending=False)
calib_long_truncated = calib_long[rank_from_end >= HORIZON]

# --- test: 
test_long = train_long_scaled[train_long_scaled['unique_id'].isin(test_ids)].sort_values(['unique_id', 'ds'])

print("Forecasting calib/test'...")
boost_calib_scaled = mlf_boost_cal.predict(h=HORIZON, new_df=calib_long_truncated)
boost_test_scaled  = mlf_boost_cal.predict(h=HORIZON, new_df=test_long)
rf_calib_scaled = mlf_rf_cal.predict(h=HORIZON, new_df=calib_long_truncated)
rf_test_scaled  = mlf_rf_cal.predict(h=HORIZON, new_df=test_long)

boost_calib = inverse_scale(boost_calib_scaled, boost_model_cols)
boost_test  = inverse_scale(boost_test_scaled, boost_model_cols)
rf_calib = inverse_scale(rf_calib_scaled, rf_model_cols)
rf_test  = inverse_scale(rf_test_scaled, rf_model_cols)

# --- export: 
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

def export_long_to_wide(df, ids, model_col, model_name, split):
    sub = df[df['unique_id'].isin(ids)].sort_values(['unique_id', 'ds'])
    wide = sub.groupby('unique_id')[model_col].apply(lambda s: s.values[:HORIZON])
    rows = [{"unique_id": uid, **{f"h{i+1}": v for i, v in enumerate(wide[uid])}} for uid in ids]
    pd.DataFrame(rows).to_csv(RESULTS_DIR / f"{split}_{model_name}.csv", index=False)

for model_col in boost_model_cols:
    export_long_to_wide(boost_calib, calib_ids, model_col, model_col, "calib")
    export_long_to_wide(boost_test, test_ids, model_col, model_col, "test")

for model_col in rf_model_cols:
    export_long_to_wide(rf_calib, calib_ids, model_col, model_col, "calib")
    export_long_to_wide(rf_test, test_ids, model_col, model_col, "test")

print("Exported:", boost_model_cols + rf_model_cols)